# Dimensionality Reduction & Clustering

Package-backed computational workflow for fitted gene-expression trajectories and dynamic-mode clustering. Publication plotting is intentionally kept out of this notebook for now.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import chronopack as cp


In [2]:
workspace_dir = Path.cwd().resolve()
if workspace_dir.name == "k562-tnfa-puromycin.hg38":
    repo_root = workspace_dir.parent
else:
    repo_root = workspace_dir

puro_dir = repo_root / "k562-tnfa-puromycin.hg38"
k562a_dir = repo_root / "k562-tnfa.hg38.rep"
lps_dir = repo_root / "thp1-k562-lps.hg38.rep"

# Use the SCE/tradeSeq artifacts generated inside this verification run.
root_puro = puro_dir
root_k562a = k562a_dir
root_lps = lps_dir

SELF_CONTAINED_OUTDIR = puro_dir / "self_contained_outputs" / "global_analysis_extended"
SELF_CONTAINED_OUTDIR.mkdir(parents=True, exist_ok=True)

{
    "root_puro": str(root_puro),
    "root_k562a": str(root_k562a),
    "root_lps": str(root_lps),
    "output": str(SELF_CONTAINED_OUTDIR),
}


{'root_puro': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa-puromycin.hg38', 'root_k562a': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa.hg38.rep', 'root_lps': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/thp1-k562-lps.hg38.rep', 'output': '/new-stg/home/alex/tmp/ChronoPack/verification_runs/manuscript_full_pipeline_20260919_165539/k562-tnfa-puromycin.hg38/self_contained_outputs/global_analysis_extended'}

## Characterize Fitted Gene Trajectories

This stage is split into alignment/standardization and descriptor calculation so the standardized trajectory matrices can be reused independently from the descriptor table.


In [3]:
alignment = cp.tl.align_gene_trajectories(
    root_puro=root_puro,
    root_k562a=root_k562a,
    root_lps=root_lps,
    output_dir=SELF_CONTAINED_OUTDIR,
    time_map={"K562a": 110, "K562_puro": 110, "K562b": 165, "THP1": 165},
    offset_map={"K562a": 0, "K562_puro": 0, "K562b": -68, "THP1": -10},
    scale_map={"THP1": 0.65},
)

char = cp.tl.compute_gene_trajectory_descriptors(
    alignment=alignment,
    output_dir=SELF_CONTAINED_OUTDIR,
)

char.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_characterization.h5ad")
{
    "adata_shape": char.adata.shape,
    "time_grid": char.metadata["time_grid"],
    "aligned_rows": None if alignment.pc_df is None else alignment.pc_df.shape[0],
    "pc_df_characterized_rows": None if char.pc_df is None else char.pc_df.shape[0],
}


[K562a] TMAX=110, scale=1, offset=0 => aligned range 0.00..110.00, resampled to 110 minute points
[K562_puro] TMAX=110, scale=1, offset=0 => aligned range 0.00..110.00, resampled to 110 minute points
[K562b] TMAX=165, scale=1, offset=-68 => aligned range 0.00..97.00, resampled to 97 minute points
[THP1] TMAX=165, scale=0.65, offset=-10 => aligned range 0.40..97.25, resampled to 97 minute points
Shortest aligned end time across keys after rescaling: 97 min
Final shared aligned minute range: 1 .. 97  (n = 97 )


R callback write-console: Loading fixed tradeSeq objects: K562_puro
  
R callback write-console: Loading required package: SingleCellExperiment
  
R callback write-console: Loading required package: SummarizedExperiment
  
R callback write-console: Loading required package: MatrixGenerics
  
R callback write-console: Loading required package: matrixStats
  
R callback write-console: 
Attaching package: 'MatrixGenerics'

  
R callback write-console: The following objects are masked from 'package:matrixStats':

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, 

{'adata_shape': (0, 6723), 'time_grid': [0.0103092783505155, 0.0206185567010309, 0.0309278350515464, 0.0412371134020619, 0.0515463917525773, 0.0618556701030928, 0.0721649484536082, 0.0824742268041237, 0.0927835051546392, 0.103092783505155, 0.11340206185567, 0.123711340206186, 0.134020618556701, 0.144329896907216, 0.154639175257732, 0.164948453608247, 0.175257731958763, 0.185567010309278, 0.195876288659794, 0.206185567010309, 0.216494845360825, 0.22680412371134, 0.237113402061856, 0.247422680412371, 0.257731958762887, 0.268041237113402, 0.278350515463918, 0.288659793814433, 0.298969072164948, 0.309278350515464, 0.319587628865979, 0.329896907216495, 0.34020618556701, 0.350515463917526, 0.360824742268041, 0.371134020618557, 0.381443298969072, 0.391752577319588, 0.402061855670103, 0.412371134020619, 0.422680412371134, 0.43298969072165, 0.443298969072165, 0.45360824742268, 0.463917525773196, 0.474226804123711, 0.484536082474227, 0.494845360824742, 0.505154639175258, 0.515463917525773, 0.525

## Cluster Gene Trajectories

The notebook explicitly defines induced and repressed gene sets, evaluates clustering metrics across K, visualizes the elbow and silhouette criteria, and derives the K choices from those metrics before assigning labels.


In [4]:
pc_char = char.pc_df.copy()
LOG2FC_THRESHOLD = 0
ALPHA = 0.25

induced_rows = pc_char.index[pc_char["peak_log2fc"] > LOG2FC_THRESHOLD].tolist()
repressed_rows = pc_char.index[pc_char["peak_log2fc"] < -LOG2FC_THRESHOLD].tolist()

induced_metrics = cp.tl.cluster_gene_trajectories(
    adata=char.adata,
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
    rows=induced_rows,
    cluster_name="induced",
    alpha=ALPHA,
)

repressed_metrics = cp.tl.cluster_gene_trajectories(
    adata=char.adata,
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
    rows=repressed_rows,
    cluster_name="repressed",
    alpha=ALPHA,
)

metrics_by_name = {
    "induced": induced_metrics.metrics["induced"].copy(),
    "repressed": repressed_metrics.metrics["repressed"].copy(),
}

def choose_k_from_metrics(df):
    k_fine = int(df.loc[df["elbow_score"].idxmax(), "K"])
    k_broad = int(df.loc[df["mean_silhouette"].idxmax(), "K"])
    return {"broad": k_broad, "fine": k_fine}

selected_k = {name: choose_k_from_metrics(df) for name, df in metrics_by_name.items()}
metrics_summary = pd.DataFrame({
    name: {
        "k_elbow_for_fine": vals["fine"],
        "k_max_mean_silhouette_for_broad": vals["broad"],
    }
    for name, vals in selected_k.items()
}).T
display(metrics_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex="col")
for row, (name, df) in enumerate(metrics_by_name.items()):
    k_fine = selected_k[name]["fine"]
    k_broad = selected_k[name]["broad"]

    axes[row, 0].plot(df["K"], df["elbow_score"], color="#2b6cb0", linewidth=1.6)
    axes[row, 0].scatter(
        [k_fine],
        [df.loc[df["K"].eq(k_fine), "elbow_score"].iloc[0]],
        color="#c2410c",
        zorder=3,
        label=f"selected fine K={k_fine}",
    )
    axes[row, 0].axvline(k_fine, color="#c2410c", linestyle="--", linewidth=1)
    axes[row, 0].set_title(f"{name}: elbow score")
    axes[row, 0].set_ylabel("score")
    axes[row, 0].legend(loc="best", frameon=False)

    axes[row, 1].plot(df["K"], df["mean_silhouette"], color="#047857", linewidth=1.6)
    axes[row, 1].scatter(
        [k_broad],
        [df.loc[df["K"].eq(k_broad), "mean_silhouette"].iloc[0]],
        color="#c2410c",
        zorder=3,
        label=f"selected broad K={k_broad}",
    )
    axes[row, 1].axvline(k_broad, color="#c2410c", linestyle="--", linewidth=1)
    axes[row, 1].set_title(f"{name}: mean silhouette")
    axes[row, 1].legend(loc="best", frameon=False)

for ax in axes[-1, :]:
    ax.set_xlabel("K")
for ax in axes.ravel():
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(SELF_CONTAINED_OUTDIR / "cluster_metric_selection.png", dpi=200, bbox_inches="tight")
plt.show()

{
    "alpha": ALPHA,
    "induced_rows": len(induced_rows),
    "repressed_rows": len(repressed_rows),
    "selected_k_from_metrics": selected_k,
    "metric_plot": str(SELF_CONTAINED_OUTDIR / "cluster_metric_selection.png"),
}


R callback write-console: 2026-09-22 11:24:14 | Starting PCA for 6723 genes x 97 trajectory features.
  
R callback write-console: 2026-09-22 11:24:15 | Finished PCA: n_pc_X=4, n_pc_Xd=5.
  
R callback write-console: 2026-09-22 11:24:15 | Starting induced clustering.
  
R callback write-console: 2026-09-22 11:26:10 | Finished induced clustering: k_elbow=15, k_silhouette=3.
  
R callback write-console: 2026-09-22 11:26:10 | Starting repressed clustering.
  
R callback write-console: 2026-09-22 11:28:20 | Finished repressed clustering: k_elbow=17, k_silhouette=3.
  
R callback write-console: 2026-09-22 11:28:22 | Writing clustering outputs.
  
R callback write-console: 2026-09-22 11:28:34 | Finished writing clustering outputs.
  


{'adata_shape': (0, 6723), 'pc_df_shape': (6723, 35), 'broad_cluster_counts': {'trend-down_1': 2215, 'trend-up': 1947, 'transient-down': 1027, 'impulse-up_1': 887, 'impulse-up_2': 379, 'trend-down_2': 268}}

### Cluster Metric Selection Plot

![Cluster metric selection](../self_contained_outputs/global_analysis_extended/cluster_metric_selection.png)


## Apply Selected K Values

The selected broad and fine K values are now applied to produce the final induced and repressed cluster assignments.


In [ ]:
induced_cluster = cp.tl.cluster_gene_trajectories(
    adata=char.adata,
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
    rows=induced_rows,
    cluster_name="induced",
    alpha=ALPHA,
    k_broad=selected_k["induced"]["broad"],
    k_fine=selected_k["induced"]["fine"],
)

repressed_cluster = cp.tl.cluster_gene_trajectories(
    adata=char.adata,
    characterization=char,
    output_dir=SELF_CONTAINED_OUTDIR,
    rows=repressed_rows,
    cluster_name="repressed",
    alpha=ALPHA,
    k_broad=selected_k["repressed"]["broad"],
    k_fine=selected_k["repressed"]["fine"],
)

cluster = cp.tl.finalize_gene_trajectory_clusters(
    adata=char.adata,
    characterization=char,
    clusterings=[induced_cluster, repressed_cluster],
    output_dir=SELF_CONTAINED_OUTDIR,
)

cluster.adata.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_clusters.h5ad")
pc_df = cluster.pc_df
pc_df.to_csv(SELF_CONTAINED_OUTDIR / "pc_df.new.csv")

{
    "adata_shape": cluster.adata.shape,
    "pc_df_shape": pc_df.shape,
    "broad_cluster_counts": pc_df["broad_cluster"].astype(str).value_counts().to_dict(),
}


## Inspect Outputs


In [5]:
display(pc_df.head())
print(pc_df.shape)
print(pc_df["broad_cluster"].value_counts())


(6723, 35)
broad_cluster
trend-down_1      2215
trend-up          1947
transient-down    1027
impulse-up_1       887
impulse-up_2       379
trend-down_2       268
Name: count, dtype: int64


,gene,cell_type,t_onset,t_peak,baseline_exp,peak_exp,peak_dlogexp,peak_dexp,peak_log2fc,auc_norm,owp,owp_ddt,mean_curv,max_curv,mean_turn,max_turn,waldStat_1,meanLogFC,pvalue_1,qval_1,PC1,PC2,PC3,PC4,cluster_broad_induced,cluster_dist_broad_induced,cluster_fine_induced,cluster_dist_fine_induced,cluster_broad_repressed,cluster_dist_broad_repressed,cluster_fine_repressed,cluster_dist_fine_repressed,broad_cluster,cluster_dist_broad,cluster_dist_fine
NFKBIA_K562a,NFKBIA,K562a,13.0,44.0,0.205383,1.669589,1.464206,4.081991,4.240605,0.660197,0.873081,0.254154,3.203454,16.999057,0.013560,0.057604,1967.377559,2.806293,0.0,0.0,-6.760465,-7.896757,1.886260,-1.602042,impulse-up_1,0.652766,1,0.237635,-1,NaN,-1,NaN,impulse-up_1,0.652766,0.237635
CXCL3_K562a,CXCL3,K562a,12.0,40.0,0.073959,0.956644,0.882685,1.526183,4.384173,0.715159,0.922997,0.245469,3.996198,27.634036,0.016469,0.092331,742.323899,2.786376,0.0,0.0,-8.009887,-6.520550,2.490377,-0.687918,impulse-up_1,0.653564,1,0.298007,-1,NaN,-1,NaN,impulse-up_1,0.653564,0.298007
CD69_K562a,CD69,K562a,13.0,41.0,0.031030,0.850327,0.819297,1.308895,5.410430,0.610903,0.810655,0.227155,4.699037,31.936844,0.019736,0.107055,602.364734,3.454632,0.0,0.0,-6.107661,-7.916383,2.446116,-1.639655,impulse-up_1,0.733679,1,0.352253,-1,NaN,-1,NaN,impulse-up_1,0.733679,0.352253
CD83_K562a,CD83,K562a,21.0,97.0,0.025817,0.361206,0.335389,0.408906,4.056148,0.423911,1.000000,0.865431,0.890947,1.705750,0.003982,0.007014,566.998711,2.128998,0.0,0.0,-10.975138,-0.392566,-0.788509,-0.192591,trend-up,0.514878,2,0.395160,-1,NaN,-1,NaN,trend-up,0.514878,0.395160
ARL4A_K562a,ARL4A,K562a,16.0,91.0,1.102629,1.487241,0.384612,1.412797,0.767366,0.558478,0.999284,0.645456,2.466544,11.254000,0.009530,0.037568,466.569370,0.365352,0.0,0.0,-10.762020,-2.443960,-0.546071,-0.291104,impulse-up_1,0.478791,3,0.375831,-1,NaN,-1,NaN,impulse-up_1,0.478791,0.375831


## Optional Reference Validation


In [6]:
import os

if os.environ.get("CHRONOPACK_VALIDATE_REFERENCE") == "1":
    reference_path = root_puro / "intmd" / "global_analysis" / "extended" / "pc_df.new.rds"
    print("Reference validation requested.")
    print("Reference path:", reference_path)
    print("Current rows:", pc_df.shape[0])
    print("Current columns:", pc_df.shape[1])
else:
    print("Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.")


Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.
